# Cloud classification with KMeans on brightness temperature

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract calibrated temperatures + navigation

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()

field = session.extract_field('''
data = loadADDEImage(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
                     size='ALL', unit='TEMP', mag=(-4, -4))
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Display', data)
''')
tb = field.masked()
print(field.shape, field.unit, '%.1f..%.1f' % (np.nanmin(tb), np.nanmax(tb)))

## 2. Cluster on temperature

In [ ]:
from sklearn.cluster import KMeans

m = np.isfinite(tb)
X = tb[m].reshape(-1, 1)
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X)

order = np.argsort(km.cluster_centers_.ravel())
labels = np.argsort(order)[km.labels_]
classes = np.full(tb.shape, np.nan)
classes[m] = labels

for c in range(5):
    sel = tb[m][labels == c]
    print('class %d: %6.1f - %6.1f K  (n=%d)' % (c, sel.min(), sel.max(), sel.size))

## 3. Plot

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(tb, cmap='inferno_r'); ax[0].set_title('Tb (K)'); ax[0].axis('off')
fig.colorbar(im, ax=ax[0], fraction=0.046)
im2 = ax[1].imshow(classes, cmap='turbo'); ax[1].set_title('KMeans classes'); ax[1].axis('off')
fig.colorbar(im2, ax=ax[1], fraction=0.046)
plt.tight_layout()

## 4. Push back, correctly geolocated

In [ ]:
from scipy.interpolate import griddata

def regrid(field, values, nlat=180, nlon=300, fill=np.nan):
    m = field.valid & np.isfinite(values)
    pts = np.column_stack([field.lats[m], field.lons[m]])
    glats = np.linspace(np.nanmax(field.lats[m]), np.nanmin(field.lats[m]), nlat)
    glons = np.linspace(np.nanmin(field.lons[m]), np.nanmax(field.lons[m]), nlon)
    GLA, GLO = np.meshgrid(glats, glons, indexing='ij')
    g = griddata(pts, values[m], (GLA, GLO), method='linear')
    return np.where(np.isfinite(g), g, fill).astype('f4'), glats, glons

grid, glats, glons = regrid(field, classes, fill=-1)
session.run('''
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setLayerLabel(label='KMeans classes from brightness temperature')
''', arrays={'g': (grid, glats, glons)})